# 01 — Retrieval from Qdrant for Financial QA

**Goal:** Hybrid retrieval (dense + sparse via RRF) from a Qdrant collection of 10-K annual report chunks, evaluated against the UDA benchmark.

**Pipeline stage:** Retrieval only — no reranking, no LLM generation.

## Cell 1 — Setup & Connection Test

In [ ]:
import os
import json
import pickle
import pandas as pd
from pathlib import Path
from qdrant_client import QdrantClient
from qdrant_client.models import (
    FieldCondition, Filter, MatchValue,
    NamedSparseVector, NamedVector,
    SearchParams, SparseVector,
    FusionQuery, Fusion, Prefetch, QueryRequest,
)
from FlagEmbedding import BGEM3FlagModel

# ── Config ──────────────────────────────────────────────
QDRANT_PATH = "./qdrant_db"       # local embedded storage (same as ingest)
COLLECTION = "annual_reports"
BGE_M3_MODEL = "BAAI/bge-m3"
TOP_K = 50  # retrieve wide, rerank later

# ── Connect to Qdrant (local/embedded mode) ─────────────
client = QdrantClient(path=QDRANT_PATH)

info = client.get_collection(COLLECTION)
print(f"Collection: {COLLECTION}")
print(f"Points:     {info.points_count}")
print(f"Vectors:    {list(info.config.params.vectors.keys())}")
print(f"Sparse:     {list(info.config.params.sparse_vectors.keys())}")
print(f"Status:     {info.status}")

In [ ]:
# ── Load BGE-M3 ────────────────────────────────────────
embed_model = BGEM3FlagModel(BGE_M3_MODEL, use_fp16=True)
print(f"BGE-M3 loaded (fp16, device: cuda)")

# Quick sanity check: encode a test query
test_out = embed_model.encode(
    ["What is the total revenue?"],
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=False,
)
print(f"Dense dim:    {len(test_out['dense_vecs'][0])}")
print(f"Sparse nnz:   {len(test_out['lexical_weights'][0])}")
print("✓ Embedding model ready")

## Cell 2 — Load Benchmark CSV

In [ ]:
# ── Load QA benchmarks ────────────────────────────────────
QA_DIR = Path("./data/qa")

df_fin = pd.read_csv(QA_DIR / "fin_qa.csv")
df_feta = pd.read_csv(QA_DIR / "feta_qa.csv")

# Combine into one dataframe
df_fin["source"] = "fin"
df_feta["source"] = "feta"
df = pd.concat([df_fin, df_feta], ignore_index=True)

# Parse doc_name → ticker + year
df[["ticker", "year"]] = df["doc_name"].str.split("_", n=1, expand=True)
df["year"] = df["year"].astype(int)

# Extract ground-truth page from q_uid  (e.g. 'ADI/2009/page_49.pdf-1' → 'ADI/2009/page_49.pdf')
df["gt_pdf"] = df["q_uid"].str.rsplit("-", n=1).str[0]

print(f"Loaded {len(df)} questions ({len(df_fin)} fin + {len(df_feta)} feta)")
print(f"Tickers: {sorted(df['ticker'].unique())}")
print(f"Years:   {sorted(df['year'].unique())}")
print()
df[["doc_name", "ticker", "year", "q_uid", "gt_pdf", "question"]].head(10)